# v00 · First Age — blockchain ledger and its own self-test

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


## The file's own `run_self_test()`

In [2]:
fa = load("PtolemyDesktop/Archive/ainulindale_first_age/FA_smnnip_hyperindex.py", "fa")
ok = fa.run_self_test()
print("self-test returned:", ok)

FA_smnnip_hyperindex — self-test
  [PASS] USKeyboardMap: charset verified (97 chars)
  [PASS] TextHyperIndex: 'Hello!' -> idx=283746489927 -> 'Hello!'
  [PASS] TextHyperIndex long: 41 chars, 270 bits
  [PASS] SphericalColor: theta=0.299 sat=0.778 rho=0.392 lum_err=0.0506 (phi is advisory/lossy)
  [PASS] MultiChannelTupper: 4ch 8×8 tile, total bits=2048, err=0.002941
  [PASS] Radial correspondence: ℝ(r=1.0)→ℂ(0.75)→ℍ(0.50)→𝕆(0.25)
  [PASS] Sedenion boundary: 𝕆 passes, 𝕊 raises
  [PASS] Polar Lagrangian: (2/π)∮[...+(1/φ)L_bias+...] = -0.053826
  [PASS] Blockchain ledger: 2-block chain verified, mastery event recorded
  Results: 9 passed, 0 failed
self-test returned: True


## The SHA-256-chained ledger: append, verify, tamper
Written to a temporary directory; the repo is not touched.

In [3]:
import tempfile, os, json
d = tempfile.mkdtemp()
L = fa.BlockchainLedger(os.path.join(d, "ledger"))
for i in range(3):
    idx, _, n = fa.TextHyperIndex.index_record("entry %d" % i)
    L.append("text", 0, i, idx, n, "R", 0.01 * i)
print("verify:", L.verify())
# tamper with block 1 on disk and verify again
p = os.path.join(d, "ledger.jsonl"); rows = [json.loads(x) for x in open(p)]
rows[1]["step"] = 99
open(p, "w").write("\n".join(json.dumps(r) for r in rows) + "\n")
print("after tamper:", L.verify())
assert L.verify()[0] is False

verify: (True, -1, 'Chain valid — 3 blocks verified')
after tamper: (False, 1, 'Chain broken at block 1: hash mismatch')
